In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from openai import OpenAI
from tqdm import tqdm
import time

def test_embedding_api():
    """测试embedding接口是否正常工作"""
    client = OpenAI(
        base_url='https://api.openai-proxy.org/v1',
        api_key='sk-BXfJ2HSgqHVq7J7aEPwN7s9NapSyCS9eFHFacQjbZF213q2z',
    )
    
    # 测试embedding接口
    response = client.embeddings.create(
        input="Hello, this is a test prompt for embedding generation.",
        model="text-embedding-3-large"  # 使用large版本获得更好质量
    )
    
    print(f"Embedding维度: {len(response.data[0].embedding)}")
    print(f"前5个embedding值: {response.data[0].embedding[:5]}")
    return client

def load_csv_data(csv_path):
    """加载CSV文件中的prompt数据"""
    df = pd.read_csv(csv_path)
    print(f"CSV文件包含 {len(df)} 行数据")
    print(f"列名: {df.columns.tolist()}")
    
    # 检查final_fixed_prompt列
    if 'final_fixed_prompt' in df.columns:
        print(f"找到 {df['final_fixed_prompt'].notna().sum()} 个有效prompt")
        return df
    else:
        raise ValueError("CSV文件中没有找到'final_fixed_prompt'列")

def batch_generate_embeddings(client, df, batch_size=100, cache_file=None, mapping_file=None):
    """批量生成embeddings并缓存，同时保存sample_ID映射"""
    
    # 检查缓存文件是否存在
    if cache_file and os.path.exists(cache_file):
        print(f"发现缓存文件 {cache_file}，正在加载...")
        with open(cache_file, 'rb') as f:
            cached_data = pickle.load(f)
            print(f"从缓存加载了 {len(cached_data)} 个embeddings")
            
            # 同时加载映射文件
            if mapping_file and os.path.exists(mapping_file):
                with open(mapping_file, 'rb') as f:
                    sample_id_mapping = pickle.load(f)
                    print(f"从缓存加载了 {len(sample_id_mapping)} 个sample_ID映射")
                    return cached_data, sample_id_mapping
            else:
                return cached_data, None
    
    # 提取有效的prompts和对应的sample_IDs
    valid_mask = df['final_fixed_prompt'].notna()
    valid_df = df[valid_mask].reset_index(drop=True)
    prompts = valid_df['final_fixed_prompt'].tolist()
    sample_ids = valid_df['sample_ID'].tolist()
    
    print(f"准备处理 {len(prompts)} 个有效prompts")
    
    embeddings = []
    sample_id_mapping = {}  # sample_ID -> embedding_index
    
    # 分批处理prompts
    for i in tqdm(range(0, len(prompts), batch_size), desc="生成embeddings"):
        batch_prompts = prompts[i:i+batch_size]
        batch_sample_ids = sample_ids[i:i+batch_size]
        
        try:
            response = client.embeddings.create(
                input=batch_prompts,
                model="text-embedding-3-large"
            )
            
            # 提取embeddings并建立映射
            for j, (embedding_data, sample_id) in enumerate(zip(response.data, batch_sample_ids)):
                embeddings.append(embedding_data.embedding)
                embedding_index = len(embeddings) - 1
                sample_id_mapping[sample_id] = embedding_index
            
            # 添加延迟避免API限制
            time.sleep(0.1)
            
        except Exception as e:
            print(f"处理批次 {i//batch_size + 1} 时出错: {e}")
            # 如果批量失败，尝试单个处理
            for prompt, sample_id in zip(batch_prompts, batch_sample_ids):
                try:
                    response = client.embeddings.create(
                        input=prompt,
                        model="text-embedding-3-large"
                    )
                    embeddings.append(response.data[0].embedding)
                    embedding_index = len(embeddings) - 1
                    sample_id_mapping[sample_id] = embedding_index
                    time.sleep(0.1)
                except Exception as e2:
                    print(f"单个prompt处理失败: {e2}")
                    embeddings.append(None)  # 用None标记失败的embedding
    
    # 保存到缓存
    if cache_file:
        print(f"保存embeddings到缓存文件: {cache_file}")
        with open(cache_file, 'wb') as f:
            pickle.dump(embeddings, f)
    
    # 保存sample_ID映射
    if mapping_file:
        print(f"保存sample_ID映射到文件: {mapping_file}")
        with open(mapping_file, 'wb') as f:
            pickle.dump(sample_id_mapping, f)
    
    return embeddings, sample_id_mapping

if __name__ == '__main__':
    print("=== 测试Embedding接口 ===")
    client = test_embedding_api()
    
    # 2. 加载CSV数据
    print("\n=== 加载CSV数据 ===")
    csv_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/curatedMetagenomicData_smart_prompts.csv"
    df = load_csv_data(csv_path)
    
    # 3. 提取prompts
    prompts = df['final_fixed_prompt'].dropna().tolist()
    print(f"准备处理 {len(prompts)} 个prompts")
    
    # 4. 批量生成embeddings
    print("\n=== 批量生成Embeddings ===")
    cache_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/prompt_embeddings_large_cache.pkl"
    mapping_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/sample_id_mapping.pkl"
    
    # 先处理前100个作为测试
    test_df = df.head(100)
    print(f"测试模式：处理前 {len(test_df)} 个样本")
    
    embeddings, sample_id_mapping = batch_generate_embeddings(
        client=client,
        df=test_df,
        batch_size=10,  # 小批次测试
        cache_file=cache_file,
        mapping_file=mapping_file
    )
    
    print(f"\n成功生成 {len([e for e in embeddings if e is not None])} 个embeddings")
    print(f"Embedding维度: {len(embeddings[0]) if embeddings[0] is not None else 'N/A'}")

In [ ]:
# 处理完整的prompt数据集
def process_full_dataset():
    """处理完整的prompt数据集"""
    print("=== 处理完整数据集 ===")
    
    # 重新加载完整数据
    csv_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/curatedMetagenomicData_smart_prompts.csv"
    df = load_csv_data(csv_path)
    
    # 提取所有prompts
    prompts = df['final_fixed_prompt'].dropna().tolist()
    print(f"准备处理 {len(prompts)} 个prompts")
    
    # 生成embeddings
    cache_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/full_prompt_embeddings_large_cache.pkl"
    mapping_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/full_sample_id_mapping.pkl"
    
    embeddings, sample_id_mapping = batch_generate_embeddings(
        client=client,
        df=df,
        batch_size=50,  # 适中的批次大小
        cache_file=cache_file,
        mapping_file=mapping_file
    )
    
    # 统计结果
    successful_embeddings = [e for e in embeddings if e is not None]
    failed_count = len(embeddings) - len(successful_embeddings)
    
    print(f"\n=== 处理完成 ===")
    print(f"总样本数: {len(df)}")
    print(f"有效prompts: {len(prompts)}")
    print(f"成功生成embeddings: {len(successful_embeddings)}")
    print(f"失败数量: {failed_count}")
    print(f"成功率: {len(successful_embeddings)/len(prompts)*100:.2f}%")
    print(f"Sample_ID映射数量: {len(sample_id_mapping) if sample_id_mapping else 0}")
    
    if successful_embeddings:
        print(f"Embedding维度: {len(successful_embeddings[0])}")
        
        # 保存为numpy数组供后续使用
        embeddings_array = np.array(successful_embeddings)
        np.save("/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/prompt_embeddings_large.npy", embeddings_array)
        print(f"Embeddings已保存为numpy数组: prompt_embeddings_large.npy")
        print(f"数组形状: {embeddings_array.shape}")
        
        # 保存sample_ID映射为JSON格式，便于查看
        if sample_id_mapping:
            import json
            json_mapping_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/sample_id_mapping.json"
            with open(json_mapping_file, 'w') as f:
                json.dump(sample_id_mapping, f, indent=2)
            print(f"Sample_ID映射已保存为JSON: sample_id_mapping.json")
        else:
            print("Sample_ID映射为None，跳过JSON保存")
    
    return embeddings, sample_id_mapping

# 取消注释下面的行来运行完整数据集处理
process_full_dataset()


In [ ]:
# 加载和使用embeddings的示例
def load_and_use_embeddings():
    """加载embeddings并展示如何使用"""
    
    # 方法1: 从numpy文件加载
    embeddings_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/prompt_embeddings_large.npy"
    if os.path.exists(embeddings_file):
        embeddings_array = np.load(embeddings_file)
        print(f"从numpy文件加载embeddings: {embeddings_array.shape}")
        
        # 展示前几个embedding
        print(f"前3个embedding的前5维:")
        for i in range(min(3, len(embeddings_array))):
            print(f"  Embedding {i}: {embeddings_array[i][:5]}")
    
    # 方法2: 从pickle缓存加载
    cache_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/full_prompt_embeddings_large_cache.pkl"
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            cached_embeddings = pickle.load(f)
        print(f"从缓存文件加载embeddings: {len(cached_embeddings)} 个")
    
    # 加载sample_ID映射
    mapping_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/full_sample_id_mapping.pkl"
    sample_id_mapping = None
    if os.path.exists(mapping_file):
        with open(mapping_file, 'rb') as f:
            sample_id_mapping = pickle.load(f)
        print(f"从缓存文件加载sample_ID映射: {len(sample_id_mapping)} 个")
    
    # 加载对应的prompts
    csv_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/curatedMetagenomicData_final_fixed_prompts.csv"
    df = pd.read_csv(csv_path)
    prompts = df['final_fixed_prompt'].dropna().tolist()
    
    print(f"\n=== Embeddings使用示例 ===")
    print(f"总prompts数量: {len(prompts)}")
    print(f"Embeddings维度: {embeddings_array.shape[1] if 'embeddings_array' in locals() else 'N/A'}")
    print(f"Sample_ID映射数量: {len(sample_id_mapping) if sample_id_mapping else 'N/A'}")
    
    # 展示sample_ID查询功能
    if sample_id_mapping:
        print(f"\n=== Sample_ID查询示例 ===")
        # 展示前5个sample_ID及其对应的embedding索引
        sample_items = list(sample_id_mapping.items())[:5]
        for sample_id, embedding_idx in sample_items:
            print(f"Sample_ID: {sample_id} -> Embedding索引: {embedding_idx}")
            
            # 如果embedding存在，显示对应的prompt
            if 'embeddings_array' in locals() and embedding_idx < len(embeddings_array):
                # 找到对应的prompt
                mask = df['sample_ID'] == sample_id
                if mask.any():
                    prompt = df[mask]['final_fixed_prompt'].iloc[0]
                    print(f"  对应Prompt: {prompt[:80]}...")
    
    # 计算embedding相似度的示例
    if 'embeddings_array' in locals() and len(embeddings_array) > 1:
        from sklearn.metrics.pairwise import cosine_similarity
        
        # 计算前10个prompt之间的相似度
        sample_embeddings = embeddings_array[:10]
        similarity_matrix = cosine_similarity(sample_embeddings)
        
        print(f"\n前10个prompt的相似度矩阵 (前5x5):")
        print(similarity_matrix[:5, :5])
        
        # 找到最相似的prompt对
        np.fill_diagonal(similarity_matrix, 0)  # 排除自己与自己的相似度
        max_sim_idx = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)
        max_similarity = similarity_matrix[max_sim_idx]
        
        print(f"\n最相似的prompt对:")
        print(f"Prompt {max_sim_idx[0]}: {prompts[max_sim_idx[0]][:100]}...")
        print(f"Prompt {max_sim_idx[1]}: {prompts[max_sim_idx[1]][:100]}...")
        print(f"相似度: {max_similarity:.4f}")

# 运行示例
load_and_use_embeddings()


In [ ]:
# Embedding数据分析：重复检测、类别相似度、2D可视化
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

def analyze_embeddings():
    """全面的embedding分析"""
    print("=== Embedding数据分析 ===")
    
    # 加载数据
    embeddings_file = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/prompt_embeddings_large.npy"
    embeddings = np.load(embeddings_file)
    print(f"Embeddings形状: {embeddings.shape}")
    
    csv_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/curatedMetagenomicData_final_fixed_prompts.csv"
    df = pd.read_csv(csv_path)
    prompts = df['final_fixed_prompt'].dropna().tolist()
    
    # 1. 重复prompt分析
    print("\n--- 重复Prompt分析 ---")
    prompt_counts = Counter(prompts)
    duplicates = {prompt: count for prompt, count in prompt_counts.items() if count > 1}
    
    print(f"总prompt数量: {len(prompts):,}")
    print(f"唯一prompt数量: {len(prompt_counts):,}")
    print(f"重复prompt数量: {len(duplicates):,}")
    print(f"重复率: {len(duplicates)/len(prompt_counts)*100:.2f}%")
    
    print(f"\n最常见的重复prompt (前5个):")
    most_common = prompt_counts.most_common(5)
    for i, (prompt, count) in enumerate(most_common, 1):
        print(f"{i}. 出现{count:3d}次: {prompt[:60]}...")
    
    # 2. 类别间相似度分析
    print("\n--- 类别间相似度分析 ---")
    
    # 计算相似度矩阵（采样计算，避免内存问题）
    n_samples = min(2000, len(embeddings))
    sample_indices = np.random.choice(len(embeddings), n_samples, replace=False)
    sample_embeddings = embeddings[sample_indices]
    sample_df = df.iloc[sample_indices]
    
    print(f"采样 {n_samples} 个样本计算相似度矩阵...")
    similarity_matrix = cosine_similarity(sample_embeddings)
    
    # 分析不同类别的相似度
    categories = ['age_category', 'gender', 'body_site', 'disease', 'study_condition', 'bmi_category']
    
    for category in categories:
        if category in sample_df.columns:
            print(f"\n{category} 类别分析:")
            
            unique_values = sample_df[category].dropna().unique()
            print(f"  唯一值: {list(unique_values)}")
            
            # 计算每个类别内部的平均相似度
            intra_similarities = []
            inter_similarities = []
            
            for value in unique_values:
                # 找到该类别值在采样数据中的位置索引
                mask = sample_df[category] == value
                indices = np.where(mask)[0]  # 获取在采样数据中的位置
                
                if len(indices) > 1:
                    # 计算类别内部相似度
                    intra_sim = []
                    for i in range(len(indices)):
                        for j in range(i+1, len(indices)):
                            sim = similarity_matrix[indices[i], indices[j]]
                            intra_sim.append(sim)
                    
                    if intra_sim:
                        intra_similarities.extend(intra_sim)
                
                # 计算与其他类别的相似度
                other_mask = sample_df[category] != value
                other_indices = np.where(other_mask)[0]
                if len(indices) > 0 and len(other_indices) > 0:
                    inter_sim = []
                    for idx1 in indices:
                        for idx2 in other_indices:
                            sim = similarity_matrix[idx1, idx2]
                            inter_sim.append(sim)
                    if inter_sim:
                        inter_similarities.extend(inter_sim)
            
            if intra_similarities and inter_similarities:
                avg_intra = np.mean(intra_similarities)
                avg_inter = np.mean(inter_similarities)
                print(f"  类别内平均相似度: {avg_intra:.4f}")
                print(f"  类别间平均相似度: {avg_inter:.4f}")
                print(f"  区分度 (内-间): {avg_intra - avg_inter:.4f}")
    
    # 3. 2D可视化
    print("\n--- 创建2D可视化 ---")
    
    # 进一步采样用于可视化
    n_viz = min(3000, len(embeddings))
    viz_indices = np.random.choice(len(embeddings), n_viz, replace=False)
    viz_embeddings = embeddings[viz_indices]
    viz_df = df.iloc[viz_indices]
    
    print(f"采样 {n_viz} 个样本进行可视化...")
    
    # 创建图形
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Embedding 2D Visualization Analysis', fontsize=16)
    
    # t-SNE可视化
    print("计算t-SNE...")
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    embeddings_tsne = tsne.fit_transform(viz_embeddings)
    
    # 按disease着色
    if 'disease' in viz_df.columns:
        diseases = viz_df['disease'].fillna('unknown')
        unique_diseases = diseases.unique()
        colors = plt.cm.Set3(np.linspace(0, 1, len(unique_diseases)))
        
        ax1 = axes[0, 0]
        for i, disease in enumerate(unique_diseases):
            mask = diseases == disease
            ax1.scatter(embeddings_tsne[mask, 0], embeddings_tsne[mask, 1], 
                       c=[colors[i]], label=disease, alpha=0.6, s=20)
        ax1.set_title('t-SNE: Colored by Disease')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.set_xlabel('t-SNE 1')
        ax1.set_ylabel('t-SNE 2')
    
    # 按age_category着色
    if 'age_category' in viz_df.columns:
        ages = viz_df['age_category'].fillna('unknown')
        unique_ages = ages.unique()
        colors = plt.cm.viridis(np.linspace(0, 1, len(unique_ages)))
        
        ax2 = axes[0, 1]
        for i, age in enumerate(unique_ages):
            mask = ages == age
            ax2.scatter(embeddings_tsne[mask, 0], embeddings_tsne[mask, 1], 
                       c=[colors[i]], label=age, alpha=0.6, s=20)
        ax2.set_title('t-SNE: Colored by Age Category')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.set_xlabel('t-SNE 1')
        ax2.set_ylabel('t-SNE 2')
    
    # 按gender着色
    if 'gender' in viz_df.columns:
        genders = viz_df['gender'].fillna('unknown')
        unique_genders = genders.unique()
        colors = plt.cm.Set1(np.linspace(0, 1, len(unique_genders)))
        
        ax3 = axes[1, 0]
        for i, gender in enumerate(unique_genders):
            mask = genders == gender
            ax3.scatter(embeddings_tsne[mask, 0], embeddings_tsne[mask, 1], 
                       c=[colors[i]], label=gender, alpha=0.6, s=20)
        ax3.set_title('t-SNE: Colored by Gender')
        ax3.legend()
        ax3.set_xlabel('t-SNE 1')
        ax3.set_ylabel('t-SNE 2')
    
    # 按body_site着色
    if 'body_site' in viz_df.columns:
        body_sites = viz_df['body_site'].fillna('unknown')
        unique_sites = body_sites.unique()
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_sites)))
        
        ax4 = axes[1, 1]
        for i, site in enumerate(unique_sites):
            mask = body_sites == site
            ax4.scatter(embeddings_tsne[mask, 0], embeddings_tsne[mask, 1], 
                       c=[colors[i]], label=site, alpha=0.6, s=20)
        ax4.set_title('t-SNE: Colored by Body Site')
        ax4.legend()
        ax4.set_xlabel('t-SNE 1')
        ax4.set_ylabel('t-SNE 2')
    
    plt.tight_layout()
    plt.savefig('/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/embedding_2d_visualization.png', 
                dpi=300, bbox_inches='tight')
    print("2D可视化已保存到: embedding_2d_visualization.png")
    plt.show()
    
    # 4. 总结报告
    print("\n" + "="*60)
    print("                    EMBEDDING 分析报告")
    print("="*60)
    
    print(f"\n📊 数据概览:")
    print(f"  • 总prompt数量: {len(prompts):,}")
    print(f"  • 唯一prompt数量: {len(prompt_counts):,}")
    print(f"  • 重复prompt数量: {len(duplicates):,}")
    print(f"  • 重复率: {len(duplicates)/len(prompt_counts)*100:.2f}%")
    
    print(f"\n🔍 重复分析:")
    for i, (prompt, count) in enumerate(most_common, 1):
        print(f"  {i}. 出现{count:3d}次: {prompt[:50]}...")
    
    print(f"\n💡 建议:")
    print(f"  • 重复prompt较多，建议去重或合并相似样本")
    print(f"  • 关注区分度高的类别作为主要条件特征")
    print(f"  • 考虑使用类别平衡采样策略")
    
    print("="*60)

# 运行分析
analyze_embeddings()


In [ ]:
# 测试Sample_ID查询功能
def test_sample_id_query():
    """测试sample_ID查询功能"""
    print("=== 测试Sample_ID查询功能 ===")
    
    # 导入必要的模块
    import sys
    sys.path.append('/data/home/wudezhi/project/school/x-meta/gutclip/models')
    from prompt_condition_encoder import create_prompt_condition_encoder
    
    # 路径配置
    embeddings_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/prompt_embeddings_large.npy"
    csv_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/curatedMetagenomicData_final_fixed_prompts.csv"
    mapping_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/full_sample_id_mapping.pkl"
    
    try:
        # 创建编码器
        print("创建Prompt条件编码器...")
        encoder = create_prompt_condition_encoder(
            embeddings_path=embeddings_path,
            csv_path=csv_path,
            mapping_path=mapping_path,
            embedding_dim=256
        )
        
        # 测试sample_ID查询功能
        if encoder.sample_id_mapping:
            print(f"\n=== Sample_ID查询测试 ===")
            print(f"可用的Sample_ID数量: {len(encoder.sample_id_mapping)}")
            
            # 随机选择几个sample_ID进行测试
            import random
            sample_ids = list(encoder.sample_id_mapping.keys())
            test_sample_ids = random.sample(sample_ids, min(5, len(sample_ids)))
            
            for sample_id in test_sample_ids:
                print(f"\n--- 测试Sample_ID: {sample_id} ---")
                
                # 获取样本信息
                sample_info = encoder.get_sample_info(sample_id)
                print(f"样本信息:")
                for key, value in sample_info.items():
                    if key != 'prompt':  # prompt太长，单独显示
                        print(f"  {key}: {value}")
                print(f"  prompt: {sample_info['prompt'][:100]}...")
                
                # 通过sample_ID编码
                sample_embedding = encoder.encode_by_sample_id(sample_id)
                print(f"Sample_ID编码形状: {sample_embedding.shape}")
                print(f"编码值范围: [{sample_embedding.min():.4f}, {sample_embedding.max():.4f}]")
                
                # 通过prompt编码进行比较
                prompt_embedding = encoder.encode(sample_info['prompt'])
                
                # 计算相似度
                import torch
                similarity = torch.cosine_similarity(
                    sample_embedding.unsqueeze(0), 
                    prompt_embedding.unsqueeze(0)
                )
                print(f"Sample_ID编码与Prompt编码的相似度: {similarity.item():.6f}")
                
                # 验证embedding索引
                embedding_idx = encoder.sample_id_mapping[sample_id]
                print(f"Embedding索引: {embedding_idx}")
                
        else:
            print("❌ Sample_ID映射未加载")
            
        print("\n✅ Sample_ID查询功能测试完成!")
        
    except Exception as e:
        print(f"❌ 测试失败: {e}")
        import traceback
        traceback.print_exc()

# 运行测试
test_sample_id_query()


In [ ]:
# Sample_ID查询功能使用示例
def sample_id_usage_example():
    """展示如何使用sample_ID查询功能"""
    print("=== Sample_ID查询功能使用示例 ===")
    
    # 导入必要的模块
    import sys
    sys.path.append('/data/home/wudezhi/project/school/x-meta/gutclip/models')
    from prompt_condition_encoder import create_prompt_condition_encoder
    import torch
    
    # 路径配置
    embeddings_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/prompt_embeddings_large.npy"
    csv_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/curatedMetagenomicData_final_fixed_prompts.csv"
    mapping_path = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/full_sample_id_mapping.pkl"
    
    # 创建编码器
    encoder = create_prompt_condition_encoder(
        embeddings_path=embeddings_path,
        csv_path=csv_path,
        mapping_path=mapping_path,
        embedding_dim=256
    )
    
    print("\n=== 使用场景示例 ===")
    
    # 场景1: 通过sample_ID直接获取embedding
    print("\n1. 通过sample_ID直接获取embedding:")
    sample_ids = list(encoder.sample_id_mapping.keys())[:3]
    for sample_id in sample_ids:
        embedding = encoder.encode_by_sample_id(sample_id)
        print(f"   Sample_ID: {sample_id} -> Embedding shape: {embedding.shape}")
    
    # 场景2: 获取样本的详细信息
    print("\n2. 获取样本的详细信息:")
    sample_id = sample_ids[0]
    sample_info = encoder.get_sample_info(sample_id)
    print(f"   Sample_ID: {sample_id}")
    print(f"   Age: {sample_info['age']}")
    print(f"   Gender: {sample_info['gender']}")
    print(f"   Disease: {sample_info['disease']}")
    print(f"   Body Site: {sample_info['body_site']}")
    print(f"   Study Condition: {sample_info['study_condition']}")
    
    # 场景3: 批量处理多个sample_ID
    print("\n3. 批量处理多个sample_ID:")
    batch_sample_ids = sample_ids
    batch_embeddings = []
    for sample_id in batch_sample_ids:
        embedding = encoder.encode_by_sample_id(sample_id)
        batch_embeddings.append(embedding)
    
    batch_tensor = torch.stack(batch_embeddings)
    print(f"   批量embedding形状: {batch_tensor.shape}")
    
    # 场景4: 计算样本间的相似度
    print("\n4. 计算样本间的相似度:")
    if len(batch_embeddings) >= 2:
        similarity_matrix = torch.cosine_similarity(
            batch_tensor.unsqueeze(1), 
            batch_tensor.unsqueeze(0), 
            dim=2
        )
        print(f"   相似度矩阵形状: {similarity_matrix.shape}")
        print(f"   相似度矩阵:")
        for i, sample_id in enumerate(batch_sample_ids):
            for j, other_sample_id in enumerate(batch_sample_ids):
                if i != j:
                    sim = similarity_matrix[i, j].item()
                    print(f"     {sample_id} vs {other_sample_id}: {sim:.4f}")
    
    # 场景5: 查找最相似的样本
    print("\n5. 查找最相似的样本:")
    if len(batch_embeddings) >= 2:
        # 以第一个样本为查询
        query_embedding = batch_embeddings[0]
        query_sample_id = batch_sample_ids[0]
        
        similarities = []
        for i, embedding in enumerate(batch_embeddings[1:], 1):
            sim = torch.cosine_similarity(
                query_embedding.unsqueeze(0), 
                embedding.unsqueeze(0)
            ).item()
            similarities.append((batch_sample_ids[i], sim))
        
        # 按相似度排序
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        print(f"   查询样本: {query_sample_id}")
        print(f"   最相似的样本:")
        for sample_id, sim in similarities:
            print(f"     {sample_id}: {sim:.4f}")
    
    print("\n✅ 使用示例完成!")

# 运行使用示例
sample_id_usage_example()


In [1]:
import os, time, json, pickle, hashlib
import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ====== 配置 ======
CSV = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/curatedMetagenomicData_smart_prompts.csv"
OUT_DIR = "/data/home/wudezhi/project/school/x-meta/datasets/raw/prompt"
MODEL = "text-embedding-3-large"
BATCH = 128

# 输出产物（统一命名）
BUNDLE_NPZ   = os.path.join(OUT_DIR, "bio_bundle_v1.npz")      # sample_id 对齐的最终阵列
ID2IDX_PKL   = os.path.join(OUT_DIR, "bio_id2idx_v1.pkl")      # sample_id -> 行号
MANIFEST_CSV = os.path.join(OUT_DIR, "bio_manifest_v1.csv")    # 审计清单（含 prompt、hash、unique 索引）

# ====== 工具 ======
def get_client():
    # 不要把 key 写死在代码里
    # export OPENAI_API_KEY="sk-xxxxx"
    client = OpenAI(
        base_url='https://api.openai-proxy.org/v1',
        api_key='sk-BXfJ2HSgqHVq7J7aEPwN7s9NapSyCS9eFHFacQjbZF213q2z',
    )
    return client
def sha16(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]



def batched(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]

def embed_texts(client, texts, model=MODEL, max_retries=5):
    """对一批文本做 embedding，带指数回退重试。返回 [np.array(num, dim)]"""
    delay = 0.5
    for attempt in range(max_retries):
        try:
            rsp = client.embeddings.create(model=model, input=texts)
            embs = [np.array(d.embedding, dtype=np.float32) for d in rsp.data]
            return np.stack(embs, axis=0)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(delay)
            delay = min(delay * 2, 8.0)

# ====== 主流程 ======
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    df = pd.read_csv(CSV)
    cols = [c.lower() for c in df.columns]

    # 标准化列名：sample_id / final_fixed_prompt
    if "sample_id" not in df.columns:
        # 兼容 "sample_ID" 或其他大小写
        for cand in ["sample_ID"]:
            if cand in df.columns:
                df = df.rename(columns={cand: "sample_id"})
                break
    assert "sample_id" in df.columns, "CSV 必须包含 sample_id 列"

    prompt_col = "smart_prompt"
    assert prompt_col in df.columns, f"CSV 必须包含 {prompt_col} 列"

    # 只取有效 prompt
    df = df[df[prompt_col].notna()].copy()
    df["sample_id"] = df["sample_id"].astype(str)
    df["smart_prompt"] = df[prompt_col].astype(str).str.replace("\n", " ").str.strip()

    # —— 去重：只按 sample_id 去重，prompt 不去重但会优化 API 调用 —— #
    print(f"原始数据: {len(df)} 行")
    
    # 按 sample_id 去重（保留第一个）
    df = df.drop_duplicates(subset=["sample_id"], keep="first")
    print(f"按 sample_id 去重后: {len(df)} 行")
    
    prompts = df["smart_prompt"].tolist()
    sample_ids = df["sample_id"].tolist()

    # 优化：使用 np.unique 找到唯一的 prompt，避免重复的 API 调用
    # 注意：这里不去重 prompt，只是优化 API 调用次数
    uniq_prompts, inverse = np.unique(prompts, return_inverse=True)
    print(f"总样本: {len(prompts)}, 唯一 prompt: {len(uniq_prompts)} (节省 {len(prompts)-len(uniq_prompts)} 次 API 调用)")

    # —— 批量编码唯一 prompt —— #
    client = get_client()
    uniq_embs = []
    for batch in tqdm(list(batched(list(uniq_prompts), BATCH)), desc="Embedding unique prompts"):
        embs = embed_texts(client, batch, model=MODEL)
        uniq_embs.append(embs)
    uniq_embs = np.concatenate(uniq_embs, axis=0)  # [U, D]  float32

    # 对齐回 CSV 行顺序
    embs_aligned = uniq_embs[inverse]              # [N, D] 与 prompts 一一对应
    assert embs_aligned.shape[0] == len(prompts)

    # —— 保存 bundle（对齐到 sample_id 的一一对应阵列）—— #
    # 简化：使用简单的一对一映射（因为已经去重）
    id2idx = {sid: i for i, sid in enumerate(sample_ids)}
    
    # 验证映射正确性
    assert len(id2idx) == len(sample_ids), "映射数量不匹配"
    assert len(set(sample_ids)) == len(sample_ids), "sample_ids 仍有重复"
    
    np.savez_compressed(
        BUNDLE_NPZ,
        sample_id=np.array(sample_ids, dtype=object),
        embedding=embs_aligned
    )
    with open(ID2IDX_PKL, "wb") as f:
        pickle.dump(id2idx, f, protocol=pickle.HIGHEST_PROTOCOL)

    # manifest 便于审计（含 prompt 与 hash，与行号一致）
    manifest = pd.DataFrame({
        "row_idx": np.arange(len(sample_ids), dtype=np.int32),
        "sample_id": sample_ids,
        "smart_prompt": prompts,
        "prompt_hash": [sha16(p) for p in prompts],
        "uniq_prompt_idx": inverse.astype(np.int32),
    })
    manifest.to_csv(MANIFEST_CSV, index=False)

    print("\n保存完成：")
    print(" -", BUNDLE_NPZ)
    print(" -", ID2IDX_PKL)
    print(" -", MANIFEST_CSV)
    print("对齐检查：", embs_aligned.shape, "dtype:", embs_aligned.dtype)

if __name__ == "__main__":
    main()

/tmp/ipykernel_270806/1642286726.py:53: DtypeWarning: Columns (9,13,14,16,17,18,19,20,21,22,23,25,27,28,29,30,33,52,56,61,62,63,64,65,66,67,68,69,70,71) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV)


原始数据: 16498 行
按 sample_id 去重后: 16463 行
总样本: 16463, 唯一 prompt: 16463 (节省 0 次 API 调用)


Embedding unique prompts: 100%|██████████| 129/129 [03:49<00:00,  1.78s/it]



保存完成：
 - /data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/bio_bundle_v1.npz
 - /data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/bio_id2idx_v1.pkl
 - /data/home/wudezhi/project/school/x-meta/datasets/raw/prompt/bio_manifest_v1.csv
对齐检查： (16463, 3072) dtype: float32
